# Segmentation and churn-risk score

**Question:** which customers should the retention team look at first, and why?

**Approach:** choose a few account and behaviour features, group customers into four easy-to-explain segments, then score every customer from 0 to 100 with a points table anyone can read. No fitted model: every point value is written down in one place and can be argued with.

**To reproduce:** put `customer_churn_sample.csv` next to this notebook and run all cells. Nothing is random except the weight-jitter test, which uses a fixed seed.

## 1. Load and check the data

In [1]:
import pandas as pd, numpy as np
pd.set_option("display.width", 140)

df = pd.read_csv("customer_churn_sample.csv")
df["churned"] = (df["Churn"] == "Yes").astype(int)

print("rows, columns:", df.shape)
print("missing values:", int(df.isna().sum().sum()))
print("duplicate customer IDs:", int(df["CustomerID"].duplicated().sum()))
gap = (df["TenureMonths"] * df["MonthlyCharges"] - df["TotalCharges"]).abs()
print("rows where TotalCharges differs from tenure x monthly charge by more than 0.05:", int((gap > 0.05).sum()))
for c in ["ContractType", "PaymentMethod", "SubscriptionType", "Churn"]:
    print(c, df[c].value_counts().to_dict())
print("tenure range:", df["TenureMonths"].min(), "to", df["TenureMonths"].max(),
      "| tickets range:", df["SupportTickets"].min(), "to", df["SupportTickets"].max())

rows, columns: (15, 12)
missing values: 0
duplicate customer IDs: 0
rows where TotalCharges differs from tenure x monthly charge by more than 0.05: 0
ContractType {'Month-to-Month': 7, 'One Year': 4, 'Two Year': 4}
PaymentMethod {'Credit Card': 6, 'UPI': 4, 'Bank Transfer': 3, 'Debit Card': 2}
SubscriptionType {'Basic': 7, 'Pro': 4, 'Enterprise': 4}
Churn {'No': 8, 'Yes': 7}
tenure range: 2 to 48 | tickets range: 0 to 6


## 2. Which features, and why

Churn rate for every candidate feature, so the choices below can be checked against the data.

In [2]:
def churn_by(frame, col):
    g = frame.groupby(col)["churned"].agg(customers="count", churned="sum")
    g["churn_rate"] = (g["churned"] / g["customers"]).round(3)
    return g

df["tenure_band"] = pd.cut(df["TenureMonths"], [-1, 6, 12, 24, 999],
                           labels=["0-6", "7-12", "13-24", "25+"])
df["ticket_band"] = pd.cut(df["SupportTickets"], [-1, 1, 2, 4, 999],
                           labels=["0-1", "2", "3-4", "5+"])
df["age_band"] = pd.cut(df["Age"], [0, 29, 39, 49, 120], labels=["<30", "30s", "40s", "50+"])

for col in ["ContractType", "tenure_band", "ticket_band", "PaymentMethod", "SubscriptionType", "Gender", "age_band"]:
    print(f"--- {col}")
    print(churn_by(df, col).to_string())

--- ContractType
                customers  churned  churn_rate
ContractType                                  
Month-to-Month          7        7         1.0
One Year                4        0         0.0
Two Year                4        0         0.0
--- tenure_band
             customers  churned  churn_rate
tenure_band                                
0-6                  4        4         1.0
7-12                 2        2         1.0
13-24                5        1         0.2
25+                  4        0         0.0
--- ticket_band
             customers  churned  churn_rate
ticket_band                                
0-1                  6        0         0.0
2                    2        0         0.0
3-4                  4        4         1.0
5+                   3        3         1.0
--- PaymentMethod
               customers  churned  churn_rate
PaymentMethod                                
Bank Transfer          3        0       0.000
Credit Card            6        

**Features used in the score**

| Feature | Why it is in | Caution |
|---|---|---|
| Contract type | Largest gap in the data: all 7 churners are month-to-month, and no one-year or two-year customer left | Overlaps with tenure, so the two cannot be separated in 15 rows |
| Tenure (months) | Churners average 7 months, retained customers 29 | Same overlap with contract |
| Support tickets | Churners raised 4.3 tickets on average, retained customers 1.0 | Tickets may be a symptom of a problem, not a cause |
| Payment method | UPI and debit card customers churn more than bank-transfer and credit-card customers | Weak evidence (2 to 6 customers per method), so it carries a small weight |
| Subscription plan | Basic and Pro churn, Enterprise does not | Strongly tied to price and contract, so it carries the smallest weight |

**Left out on purpose:** age and gender. Younger customers churn more here, but using age or gender to decide who gets a retention offer is hard to justify and the sample is too small to trust the pattern. Customer ID and total charges add nothing (total charges is just tenure times monthly charge).

## 3. Four behavioural segments

In [3]:
SEGMENTS = [
    "S1 Early month-to-month",        # month-to-month, 12 months or less
    "S2 Established month-to-month",  # month-to-month, more than 12 months
    "S3 Committed, building tenure",  # one- or two-year contract, 24 months or less
    "S4 Committed, long tenure",      # one- or two-year contract, more than 24 months
]

def segment(contract, tenure):
    if contract == "Month-to-Month":
        return SEGMENTS[0] if tenure <= 12 else SEGMENTS[1]
    return SEGMENTS[2] if tenure <= 24 else SEGMENTS[3]

df["segment"] = [segment(c, t) for c, t in zip(df["ContractType"], df["TenureMonths"])]
print(df["segment"].value_counts().reindex(SEGMENTS).to_string())
assert df["segment"].isin(SEGMENTS).all() and len(df) == df["segment"].value_counts().sum()

segment
S1 Early month-to-month          6
S2 Established month-to-month    1
S3 Committed, building tenure    4
S4 Committed, long tenure        4


The split follows the two strongest signals (contract and tenure) so each segment can be described in one sentence. Every customer lands in exactly one segment. S2 holds a single customer, so treat its numbers as an anecdote.

## 4. The risk score

Five rules, each adding points. Maximum is 100. Higher means more like the customers who left.

| Rule | Points |
|---|---|
| Contract | month-to-month 40, one year 10, two year 0 |
| Tenure | 6 months or less 25, 7 to 12 months 15, 13 to 24 months 5, over 24 months 0 |
| Support tickets | 0 to 1: 0, 2: 5, 3 to 4: 15, 5 or more: 20 |
| Payment method | UPI 10, debit card 10, credit card 3, bank transfer 0 |
| Plan | Basic 5, Pro 2, Enterprise 0 |

**Tiers:** High is 60 or more, Medium is 30 to 59, Low is under 30.

The point values were set by looking at this same 15-row table, so the score describes this sample. It has not been tested on customers it has not seen.

In [4]:
CONTRACT_PTS = {"Month-to-Month": 40, "One Year": 10, "Two Year": 0}
PAYMENT_PTS  = {"UPI": 10, "Debit Card": 10, "Credit Card": 3, "Bank Transfer": 0}
PLAN_PTS     = {"Basic": 5, "Pro": 2, "Enterprise": 0}
TENURE_PTS   = [(6, 25), (12, 15), (24, 5)]       # (upper bound, points); above the last bound = 0
TICKET_PTS   = [(1, 0), (2, 5), (4, 15)]          # (upper bound, points); above the last bound = 20
TICKET_TOP, TENURE_TOP = 20, 0
HIGH_AT, MEDIUM_AT = 60, 30

def _band(value, table, top):
    for upper, pts in table:
        if value <= upper:
            return pts
    return top

def _lookup(table, key, name):
    if key not in table:
        raise ValueError(f"unknown {name}: {key!r}")
    return table[key]

def risk_breakdown(contract, tenure, tickets, payment, plan):
    """Return the points from each rule. Raises ValueError on invalid input."""
    if tenure is None or tickets is None or tenure < 0 or tickets < 0:
        raise ValueError("tenure and tickets must be zero or more")
    return {
        "contract": _lookup(CONTRACT_PTS, contract, "contract"),
        "tenure":   _band(tenure, TENURE_PTS, TENURE_TOP),
        "tickets":  _band(tickets, TICKET_PTS, TICKET_TOP),
        "payment":  _lookup(PAYMENT_PTS, payment, "payment method"),
        "plan":     _lookup(PLAN_PTS, plan, "plan"),
    }

def risk_score(*args, **kwargs):
    return sum(risk_breakdown(*args, **kwargs).values())

def risk_tier(score):
    return "High" if score >= HIGH_AT else "Medium" if score >= MEDIUM_AT else "Low"

parts = df.apply(lambda r: risk_breakdown(r.ContractType, r.TenureMonths, r.SupportTickets,
                                          r.PaymentMethod, r.SubscriptionType), axis=1, result_type="expand")
parts.columns = [f"pts_{c}" for c in parts.columns]
df = pd.concat([df, parts], axis=1)
df["risk_score"] = parts.sum(axis=1)
df["risk_tier"] = df["risk_score"].map(risk_tier)

cols = ["CustomerID", "segment", "ContractType", "TenureMonths", "SupportTickets", "PaymentMethod",
        "SubscriptionType", "risk_score", "risk_tier", "Churn"]
print(df.sort_values("risk_score", ascending=False)[cols].to_string(index=False))

CustomerID                       segment   ContractType  TenureMonths  SupportTickets PaymentMethod SubscriptionType  risk_score risk_tier Churn
 CUST-1003       S1 Early month-to-month Month-to-Month             6               5           UPI            Basic         100      High   Yes
 CUST-1014       S1 Early month-to-month Month-to-Month             2               5           UPI            Basic         100      High   Yes
 CUST-1010       S1 Early month-to-month Month-to-Month             4               4           UPI            Basic          95      High   Yes
 CUST-1007       S1 Early month-to-month Month-to-Month             3               6   Credit Card            Basic          93      High   Yes
 CUST-1005       S1 Early month-to-month Month-to-Month             8               4    Debit Card              Pro          82      High   Yes
 CUST-1001       S1 Early month-to-month Month-to-Month            12               3   Credit Card            Basic          78  

## 5. Segment table

In [5]:
seg = df.groupby("segment").agg(
    customers=("CustomerID", "count"),
    churned=("churned", "sum"),
    mrr=("MonthlyCharges", "sum"),
    avg_tenure=("TenureMonths", "mean"),
    avg_tickets=("SupportTickets", "mean"),
    avg_risk_score=("risk_score", "mean"),
).reindex(SEGMENTS)
seg["churn_rate"] = seg["churned"] / seg["customers"]
seg["mrr_lost"] = df[df.churned == 1].groupby("segment")["MonthlyCharges"].sum().reindex(SEGMENTS).fillna(0)
seg = seg[["customers", "churned", "churn_rate", "mrr", "mrr_lost", "avg_tenure", "avg_tickets", "avg_risk_score"]].round(2)
seg

,customers,churned,churn_rate,mrr,mrr_lost,avg_tenure,avg_tickets,avg_risk_score
segment,,,,,,,,
S1 Early month-to-month,6,6,1.0,329.94,329.94,5.83,4.5,91.33
S2 Established month-to-month,1,1,1.0,79.99,79.99,14.00,3.0,72.00
"S3 Committed, building tenure",4,0,0.0,259.96,0.00,19.75,1.5,24.25
"S4 Committed, long tenure",4,0,0.0,599.96,0.00,38.50,0.5,2.25


## 6. Does the score separate churners from stayers?

Two checks: churn rate by tier, and AUC (the chance a randomly chosen churner scores higher than a randomly chosen stayer; 0.5 is a coin flip, 1.0 is perfect). Because the points were set on this same data, a high AUC is expected and is **not** evidence the score will work on new customers.

In [6]:
def auc(scores, labels):
    s = np.asarray(scores, float); y = np.asarray(labels)
    pos, neg = s[y == 1], s[y == 0]
    wins = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return wins / (len(pos) * len(neg))

tier = df.groupby("risk_tier")["churned"].agg(customers="count", churned="sum").reindex(["High", "Medium", "Low"])
tier["churn_rate"] = (tier["churned"] / tier["customers"]).round(3)
print(tier.to_string())
print()
print("AUC:", round(auc(df["risk_score"], df["churned"]), 3))
print("lowest churner score :", int(df.loc[df.churned == 1, "risk_score"].min()))
print("highest stayer score :", int(df.loc[df.churned == 0, "risk_score"].max()))

           customers  churned  churn_rate
risk_tier                                
High               7        7         1.0
Medium             1        0         0.0
Low                7        0         0.0

AUC: 1.0
lowest churner score : 72
highest stayer score : 35


## 7. Edge cases and tests

Every case states the expected result before running. An `assert` fails the notebook if the score ever behaves differently, so the table also works as a regression test.

In [7]:
cases = [
    # name, (contract, tenure, tickets, payment, plan), expected score, expected tier
    ("Worst case: new, month-to-month, many tickets, UPI, Basic", ("Month-to-Month", 0, 10, "UPI", "Basic"), 100, "High"),
    ("Best case: long two-year Enterprise, no tickets, bank transfer", ("Two Year", 60, 0, "Bank Transfer", "Enterprise"), 0, "Low"),
    ("Tenure boundary: 6 months", ("Two Year", 6, 0, "Bank Transfer", "Enterprise"), 25, "Low"),
    ("Tenure boundary: 7 months", ("Two Year", 7, 0, "Bank Transfer", "Enterprise"), 15, "Low"),
    ("Tenure boundary: 12 months", ("Two Year", 12, 0, "Bank Transfer", "Enterprise"), 15, "Low"),
    ("Tenure boundary: 13 months", ("Two Year", 13, 0, "Bank Transfer", "Enterprise"), 5, "Low"),
    ("Tenure boundary: 24 months", ("Two Year", 24, 0, "Bank Transfer", "Enterprise"), 5, "Low"),
    ("Tenure boundary: 25 months", ("Two Year", 25, 0, "Bank Transfer", "Enterprise"), 0, "Low"),
    ("Tickets boundary: 1 ticket", ("Two Year", 60, 1, "Bank Transfer", "Enterprise"), 0, "Low"),
    ("Tickets boundary: 2 tickets", ("Two Year", 60, 2, "Bank Transfer", "Enterprise"), 5, "Low"),
    ("Tickets boundary: 4 tickets", ("Two Year", 60, 4, "Bank Transfer", "Enterprise"), 15, "Low"),
    ("Tickets boundary: 5 tickets", ("Two Year", 60, 5, "Bank Transfer", "Enterprise"), 20, "Low"),
    ("Tickets far above range: 100 tickets", ("Two Year", 60, 100, "Bank Transfer", "Enterprise"), 20, "Low"),
    ("Loyal but frustrated: two-year, 40 months, 8 tickets, UPI, Basic", ("Two Year", 40, 8, "UPI", "Basic"), 35, "Medium"),
    ("Brand-new on a two-year contract, no tickets", ("Two Year", 1, 0, "Bank Transfer", "Enterprise"), 25, "Low"),
    ("Long-tenure month-to-month, quiet account", ("Month-to-Month", 60, 0, "Bank Transfer", "Enterprise"), 40, "Medium"),
    ("Tier boundary: exactly 60 points is High", ("Month-to-Month", 13, 0, "UPI", "Basic"), 60, "High"),
    ("Tier boundary: exactly 30 points is Medium", ("One Year", 13, 0, "UPI", "Basic"), 30, "Medium"),
]
rows = []
for name, args, exp_score, exp_tier in cases:
    got = risk_score(*args); got_tier = risk_tier(got)
    rows.append((name, got, exp_score, got_tier, exp_tier, "PASS" if (got, got_tier) == (exp_score, exp_tier) else "FAIL"))
res = pd.DataFrame(rows, columns=["case", "score", "expected_score", "tier", "expected_tier", "result"])
print(res.to_string(index=False))
assert (res["result"] == "PASS").all(), "an edge case changed"

                                                            case  score  expected_score   tier expected_tier result
       Worst case: new, month-to-month, many tickets, UPI, Basic    100             100   High          High   PASS
  Best case: long two-year Enterprise, no tickets, bank transfer      0               0    Low           Low   PASS
                                       Tenure boundary: 6 months     25              25    Low           Low   PASS
                                       Tenure boundary: 7 months     15              15    Low           Low   PASS
                                      Tenure boundary: 12 months     15              15    Low           Low   PASS
                                      Tenure boundary: 13 months      5               5    Low           Low   PASS
                                      Tenure boundary: 24 months      5               5    Low           Low   PASS
                                      Tenure boundary: 25 months      0 

In [8]:
# Invalid input must fail loudly instead of returning a silent number.
bad_inputs = {
    "negative tenure": ("Two Year", -1, 0, "UPI", "Basic"),
    "negative tickets": ("Two Year", 5, -2, "UPI", "Basic"),
    "missing tenure": ("Two Year", None, 0, "UPI", "Basic"),
    "unknown contract": ("Weekly", 5, 0, "UPI", "Basic"),
    "unknown payment method": ("Two Year", 5, 0, "Cash", "Basic"),
    "unknown plan": ("Two Year", 5, 0, "UPI", "Platinum"),
}
for name, args in bad_inputs.items():
    try:
        risk_score(*args); outcome = "NO ERROR (bug)"
    except ValueError as e:
        outcome = f"ValueError: {e}"
    print(f"{name:24s} -> {outcome}")
    assert outcome.startswith("ValueError")

# Properties that should always hold.
rng = np.random.default_rng(0)
for _ in range(500):
    c = rng.choice(list(CONTRACT_PTS)); p = rng.choice(list(PAYMENT_PTS)); s = rng.choice(list(PLAN_PTS))
    t = int(rng.integers(0, 80)); k = int(rng.integers(0, 15))
    base = risk_score(c, t, k, p, s)
    assert 0 <= base <= 100
    assert risk_score(c, t, k + 1, p, s) >= base, "more tickets must never lower the score"
    assert risk_score(c, max(t - 1, 0), k, p, s) >= base, "shorter tenure must never lower the score"
    assert base == risk_score(c, t, k, p, s), "same input must give same output"
print("range, monotonicity and repeatability checks passed on 500 random customers")
assert (df["risk_score"] == [risk_score(*a) for a in zip(df.ContractType, df.TenureMonths, df.SupportTickets, df.PaymentMethod, df.SubscriptionType)]).all()
print("table scores match the function for all", len(df), "customers")

negative tenure          -> ValueError: tenure and tickets must be zero or more
negative tickets         -> ValueError: tenure and tickets must be zero or more
missing tenure           -> ValueError: tenure and tickets must be zero or more
unknown contract         -> ValueError: unknown contract: 'Weekly'
unknown payment method   -> ValueError: unknown payment method: 'Cash'
unknown plan             -> ValueError: unknown plan: 'Platinum'
range, monotonicity and repeatability checks passed on 500 random customers
table scores match the function for all 15 customers


In [9]:
# Every possible customer profile (3 contracts x 4 tenure bands x 4 ticket bands x 4 payment x 3 plans).
import itertools
tenure_probe = [0, 7, 13, 25]       # one tenure from each band
ticket_probe = [0, 2, 3, 5]         # one ticket count from each band
all_scores = sorted({risk_score(c, t, k, p, s)
                     for c, t, k, p, s in itertools.product(CONTRACT_PTS, tenure_probe, ticket_probe, PAYMENT_PTS, PLAN_PTS)})
profiles = len(CONTRACT_PTS) * len(tenure_probe) * len(ticket_probe) * len(PAYMENT_PTS) * len(PLAN_PTS)
print("profiles checked:", profiles)
print("lowest / highest possible score:", min(all_scores), "/", max(all_scores))
print("highest Medium-or-below score:", max(s for s in all_scores if s < HIGH_AT),
      "| lowest High score:", min(s for s in all_scores if s >= HIGH_AT))
print("highest Low score:", max(s for s in all_scores if s < MEDIUM_AT),
      "| lowest Medium score:", min(s for s in all_scores if s >= MEDIUM_AT))
assert min(all_scores) == 0 and max(all_scores) == 100

profiles checked: 576
lowest / highest possible score: 0 / 100
highest Medium-or-below score: 58 | lowest High score: 60
highest Low score: 28 | lowest Medium score: 30


## 8. How fragile are the point values?

Each point value is multiplied by a random factor between 0.7 and 1.3 (2,000 draws, fixed seed). If the ranking only works for the exact numbers chosen, it will break here.

In [10]:
rng = np.random.default_rng(42)
keys = {"contract": CONTRACT_PTS, "payment": PAYMENT_PTS, "plan": PLAN_PTS}
base_cols = ["pts_contract", "pts_tenure", "pts_tickets", "pts_payment", "pts_plan"]
aucs, top7 = [], []
for _ in range(2000):
    w = rng.uniform(0.7, 1.3, size=5)
    s = (df[base_cols].values * w).sum(axis=1)
    aucs.append(auc(s, df["churned"]))
    top = np.argsort(-s)[: int(df["churned"].sum())]
    top7.append(set(df.index[top]) == set(df.index[df.churned == 1]))
aucs = np.array(aucs)
print("AUC across 2,000 jittered weightings: min", round(aucs.min(), 3), "| share equal to 1.0:", round((aucs == 1).mean(), 3))
print("share of draws where the 7 highest scores are exactly the 7 churners:", round(np.mean(top7), 3))

AUC across 2,000 jittered weightings: min 1.0 | share equal to 1.0: 1.0
share of draws where the 7 highest scores are exactly the 7 churners: 1.0


## 9. Who to look at now: active customers by score

In [11]:
watch = df[df.churned == 0].sort_values("risk_score", ascending=False)[
    ["CustomerID", "segment", "ContractType", "TenureMonths", "SupportTickets", "PaymentMethod",
     "SubscriptionType", "MonthlyCharges", "risk_score", "risk_tier"]]
print(watch.to_string(index=False))
print()
print("active customers by tier:", watch["risk_tier"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0).astype(int).to_dict())

CustomerID                       segment ContractType  TenureMonths  SupportTickets PaymentMethod SubscriptionType  MonthlyCharges  risk_score risk_tier
 CUST-1006 S3 Committed, building tenure     One Year            18               2           UPI            Basic           49.99          35    Medium
 CUST-1009 S3 Committed, building tenure     One Year            15               2   Credit Card              Pro           79.99          25       Low
 CUST-1013 S3 Committed, building tenure     One Year            22               1 Bank Transfer            Basic           49.99          20       Low
 CUST-1002 S3 Committed, building tenure     One Year            24               1 Bank Transfer              Pro           79.99          17       Low
 CUST-1011     S4 Committed, long tenure     Two Year            30               0   Credit Card       Enterprise          149.99           3       Low
 CUST-1004     S4 Committed, long tenure     Two Year            36               

## 10. What a retention effort could be worth

The sample has no active high-risk customers, so the value sits in **preventing the pattern in future sign-ups**: new customers who start month-to-month. The save rates below are planning assumptions to test in a pilot. They are not estimated from this data.

In [12]:
s1 = df[df.segment == SEGMENTS[0]]
avg_mrr = s1["MonthlyCharges"].mean()
print("S1 customers:", len(s1), "| churned:", int(s1.churned.sum()), "| average monthly charge:", round(avg_mrr, 2))
cohort = 10
at_risk = cohort * avg_mrr
print(f"Per {cohort} new month-to-month sign-ups at this mix: {at_risk:,.2f} of monthly revenue is exposed")
scen = pd.DataFrame({"assumed save rate": ["20%", "30%", "40%"], "rate": [0.2, 0.3, 0.4]})
scen["monthly revenue kept"] = (at_risk * scen["rate"]).round(2)
scen["a year"] = (scen["monthly revenue kept"] * 12).round(2)
print(scen.drop(columns="rate").to_string(index=False))
print()
print("If the true early churn rate is lower than the 100% seen here, the amounts shrink in proportion.")

S1 customers: 6 | churned: 6 | average monthly charge: 54.99
Per 10 new month-to-month sign-ups at this mix: 549.90 of monthly revenue is exposed
assumed save rate  monthly revenue kept  a year
              20%                109.98 1319.76
              30%                164.97 1979.64
              40%                219.96 2639.52

If the true early churn rate is lower than the 100% seen here, the amounts shrink in proportion.


## 11. Export

In [13]:
out_cols = ["CustomerID", "segment", "ContractType", "TenureMonths", "SupportTickets", "PaymentMethod",
            "SubscriptionType", "MonthlyCharges", "pts_contract", "pts_tenure", "pts_tickets",
            "pts_payment", "pts_plan", "risk_score", "risk_tier", "Churn"]
df[out_cols].sort_values("risk_score", ascending=False).to_csv("customer_risk_scores.csv", index=False)
seg.reset_index().to_csv("segment_table.csv", index=False)
res.to_csv("edge_case_results.csv", index=False)
print("wrote customer_risk_scores.csv, segment_table.csv, edge_case_results.csv")

wrote customer_risk_scores.csv, segment_table.csv, edge_case_results.csv


## Limitations

- 15 customers is far too few to estimate rates. One customer moves the overall churn rate by 7 points and a four-customer segment's rate by 25 points.
- Contract and tenure overlap completely (every month-to-month customer has 14 months of tenure or less, every other customer has 15 or more), so the data cannot say which one matters.
- The point values were chosen after looking at this data. Test the score on a fresh set of customers before using it to decide who gets an offer.
- No cost, discount or contact-history fields exist, so the notebook cannot say what an intervention costs or whether one would work.